In [3]:
# ─────────────────────────────────────────────────────────
# PART 1B — CLEANING & SCHEMA DESIGN
# ─────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import ast
import re
from rapidfuzz import fuzz, process
import warnings
warnings.filterwarnings("ignore")

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", "{:.1f}".format)

print("Libraries loaded.")

Libraries loaded.


In [4]:
# Load raw data — always start from raw, never from a previous cleaned version
df_raw = pd.read_excel("data/SampleDateExtract.xlsx")
df = df_raw.copy()

print(f"Raw data loaded: {df.shape[0]} rows x {df.shape[1]} columns")

Raw data loaded: 1000 rows x 18 columns


In [5]:
# ─────────────────────────────────────────────────────────
# SECTION 2 — PARSE LIST COLUMNS
# ─────────────────────────────────────────────────────────

LIST_COLS = [
    "indications",
    "interventions_drugs", 
    "drugs_datalake",
    "main_technologies",
    "specific_technologies",
    "target_names",
    "target_abbreviations"
]

def safe_parse(val):
    """
    Safely parse a stringified list into a real Python list.
    Returns empty list if parsing fails or value is null.
    """
    if pd.isna(val) or val in ("", "[]", "nan"):
        return []
    try:
        result = ast.literal_eval(val)
        if isinstance(result, list):
            return result
        return []
    except:
        return []

# Parse all list columns
for col in LIST_COLS:
    df[col] = df[col].apply(safe_parse)
    
print("List columns parsed.")
print("\nVerification — first row:")
for col in LIST_COLS:
    print(f"\n  [{col}]")
    print(f"  {df[col].iloc[0]}")
    print(f"  type: {type(df[col].iloc[0])}")

List columns parsed.

Verification — first row:

  [indications]
  ['Anorectal Cancer', 'Epithelial Neoplasm']
  type: <class 'list'>

  [interventions_drugs]
  ['Pembrolizumab', 'Lenvatinib']
  type: <class 'list'>

  [drugs_datalake]
  ['B936', 'B261']
  type: <class 'list'>

  [main_technologies]
  [['Antibody'], ['Small Molecule']]
  type: <class 'list'>

  [specific_technologies]
  [['Monoclonal Antibody'], ['Small Molecule']]
  type: <class 'list'>

  [target_names]
  [['Programmed cell death protein 1'], ['Fibroblast growth factor receptor 1', 'Fibroblast growth factor receptor 2', 'Fibroblast growth factor receptor 3', 'Fibroblast growth factor receptor 4', 'Platelet-derived growth factor receptor alpha', 'Stem cell growth factor receptor', 'Tyrosine-protein kinase receptor RET', 'Vascular endothelial growth factor receptor 1', 'Vascular endothelial growth factor receptor 2', 'Vascular endothelial growth factor receptor 3']]
  type: <class 'list'>

  [target_abbreviations]
  [[

In [6]:
# ─────────────────────────────────────────────────────────
# SECTION 3 — DERIVE BASIC FIELDS
# ─────────────────────────────────────────────────────────

# 3A — phase_int
# Map phase string to numeric value
PHASE_MAP = {
    "EARLY_PHASE1"  : 0.5,
    "PHASE1"        : 1.0,
    "PHASE1/PHASE2" : 1.5,
    "PHASE2"        : 2.0,
    "PHASE2/PHASE3" : 2.5,
    "PHASE3"        : 3.0,
    "PHASE4"        : 4.0
}

df["phase_int"] = df["phase"].map(PHASE_MAP)

print("3A. PHASE INTEGER")
print("=" * 40)
print(df[["phase", "phase_int"]].value_counts().to_string())

3A. PHASE INTEGER
phase          phase_int
PHASE2         2.0          402
PHASE1         1.0          253
PHASE1/PHASE2  1.5          126
PHASE3         3.0          124
PHASE4         4.0           25
PHASE2/PHASE3  2.5           17
EARLY_PHASE1   0.5           13


In [7]:
# 3B — trial_type
# Standard = has a proper phase recorded
# Non-standard = missing phase (surgical, anaesthesia, nutritional trials)
df["trial_type"] = df["phase"].apply(
    lambda x: "standard" if pd.notna(x) else "non_standard"
)

print("3B. TRIAL TYPE")
print("=" * 40)
print(df["trial_type"].value_counts().to_string())

3B. TRIAL TYPE
trial_type
standard        960
non_standard     40


In [8]:
# 3C — trial_duration_days
# How long did the trial run (or is planned to run)
df["trial_duration_days"] = (
    df["completion_date"] - df["start_date"]
).dt.days

print("3C. TRIAL DURATION (days)")
print("=" * 40)
print(f"Mean   : {df['trial_duration_days'].mean():.0f} days")
print(f"Median : {df['trial_duration_days'].median():.0f} days")
print(f"Min    : {df['trial_duration_days'].min():.0f} days")
print(f"Max    : {df['trial_duration_days'].max():.0f} days")
print(f"Missing: {df['trial_duration_days'].isna().sum()}")

3C. TRIAL DURATION (days)
Mean   : 1621 days
Median : 1352 days
Min    : 0 days
Max    : 11137 days
Missing: 54


In [9]:
# 3D — start_year
df["start_year"] = df["start_date"].dt.year

print("3D. START YEAR")
print("=" * 40)
print(df["start_year"].value_counts().sort_index().to_string())

3D. START YEAR
start_year
1986.0     1
1991.0     1
1993.0     2
1994.0     2
1995.0     2
1996.0     1
1997.0     7
1998.0    11
1999.0     7
2000.0    13
2001.0    15
2002.0     7
2003.0    23
2004.0    20
2005.0    42
2006.0    34
2007.0    35
2008.0    32
2009.0    37
2010.0    35
2011.0    25
2012.0    36
2013.0    32
2014.0    43
2015.0    49
2016.0    43
2017.0    33
2018.0    50
2019.0    45
2020.0    48
2021.0    70
2022.0    57
2023.0    61
2024.0    47
2025.0    28
2026.0     1


In [12]:
# Check trials with 0 duration
print("TRIALS WITH 0 DURATION")
print("=" * 40)
zero_duration = df[df["trial_duration_days"] == 0]
print(f"Count: {len(zero_duration)}")
print(zero_duration[["nct_id", "start_date", "completion_date", 
                       "recruitment_status", "enrollment"]].to_string())

# Check trials with very long duration (> 20 years)
print("\nTRIALS WITH DURATION > 20 YEARS")
print("=" * 40)
long_duration = df[df["trial_duration_days"] > 7300]
print(f"Count: {len(long_duration)}")
print(long_duration[["nct_id", "start_date", "completion_date",
                      "recruitment_status", "enrollment"]].to_string())

row = df[df["nct_id"] == "NCT06358638"].iloc[0]
print(f"Title      : {row['brief_title']}")
print(f"Indication : {row['indications']}")
print(f"Drugs      : {row['interventions_drugs']}")
print(f"Phase      : {row['phase']}")

TRIALS WITH 0 DURATION
Count: 3
          nct_id start_date completion_date recruitment_status  enrollment
17   NCT06682637 2025-03-26      2025-03-26          WITHDRAWN         0.0
177  NCT04826016 2022-03-16      2022-03-16          WITHDRAWN         0.0
416  NCT02570958 2017-10-31      2017-10-31          WITHDRAWN         0.0

TRIALS WITH DURATION > 20 YEARS
Count: 1
         nct_id start_date completion_date recruitment_status  enrollment
64  NCT06358638 2024-04-03      2054-09-30         RECRUITING        12.0
Title      : Sickle Cell Disease Transplant Using a Nonmyeloablative Approach for Patients With Anti-donor Red Cell AntibodY
Indication : ['Sickle Cell Disease']
Drugs      : ['Daratumumab', 'Alemtuzumab', 'Sirolimus', 'Total-Body Irradiation']
Phase      : PHASE2


In [11]:
# Keep original decimal version as phase_int
# Add a clean whole number version as well
PHASE_MAP_CLEAN = {
    "EARLY_PHASE1"  : 1,
    "PHASE1"        : 1,
    "PHASE1/PHASE2" : 2,
    "PHASE2"        : 2,
    "PHASE2/PHASE3" : 3,
    "PHASE3"        : 3,
    "PHASE4"        : 4
}

df["phase_numeric"] = df["phase"].map(PHASE_MAP_CLEAN)

print("Both phase columns created:")
print(df[["phase", "phase_int", "phase_numeric"]].drop_duplicates().dropna().sort_values("phase_int").to_string())

Both phase columns created:
             phase  phase_int  phase_numeric
40    EARLY_PHASE1        0.5            1.0
2           PHASE1        1.0            1.0
15   PHASE1/PHASE2        1.5            2.0
0           PHASE2        2.0            2.0
104  PHASE2/PHASE3        2.5            3.0
9           PHASE3        3.0            3.0
50          PHASE4        4.0            4.0


In [13]:
# ─────────────────────────────────────────────────────────
# SECTION 4 — LOGICAL VALIDATION CHECKS
# ─────────────────────────────────────────────────────────

print("LOGICAL VALIDATION CHECKS")
print("=" * 60)

validation_log = []  # we'll collect all issues here

# 4A — COMPLETED trials with 0 ACTUAL enrollment
mask = (
    (df["recruitment_status"] == "COMPLETED") &
    (df["enrollment"] == 0) &
    (df["enrollment_type"] == "ACTUAL")
)
flagged = df[mask]
validation_log.append({
    "check"  : "COMPLETED + 0 ACTUAL enrollment",
    "count"  : len(flagged),
    "action" : "Flag as suspicious — trial marked complete but no patients enrolled"
})
print(f"\n4A. COMPLETED trials with 0 ACTUAL enrollment: {len(flagged)}")
if len(flagged) > 0:
    print(flagged[["nct_id", "brief_title", "recruitment_status", 
                    "enrollment", "enrollment_type"]].to_string())

LOGICAL VALIDATION CHECKS

4A. COMPLETED trials with 0 ACTUAL enrollment: 0


In [15]:
# 4B — primary_completion_date AFTER completion_date
mask = (
    (df["primary_completion_date"].notna()) &
    (df["completion_date"].notna()) &
    (df["primary_completion_date"] > df["completion_date"])
)
flagged = df[mask]
validation_log.append({
    "check"  : "primary_completion_date > completion_date",
    "count"  : len(flagged),
    "action" : "Flag as inconsistent — primary completion should never be after full completion"
})
print(f"4B. primary_completion_date after completion_date: {len(flagged)}")
if len(flagged) > 0:
    print(flagged[["nct_id", "primary_completion_date", 
                    "completion_date"]].to_string())

4B. primary_completion_date after completion_date: 1
          nct_id primary_completion_date completion_date
614  NCT01680965              2020-08-31      2020-08-30


In [16]:
# 4C — Missing enrollment_type but enrollment value exists
mask = (
    (df["enrollment_type"].isna()) &
    (df["enrollment"].notna())
)
flagged = df[mask]
validation_log.append({
    "check"  : "enrollment value present but type missing",
    "count"  : len(flagged),
    "action" : "Flag — cannot interpret enrollment number without knowing if actual or estimated"
})
print(f"4C. Enrollment value present but type missing: {len(flagged)}")

4C. Enrollment value present but type missing: 18


In [17]:
# 4D — COMPLETED trials with ESTIMATED enrollment
mask = (
    (df["recruitment_status"] == "COMPLETED") &
    (df["enrollment_type"] == "ESTIMATED")
)
flagged = df[mask]
validation_log.append({
    "check"  : "COMPLETED + ESTIMATED enrollment",
    "count"  : len(flagged),
    "action" : "Note — enrollment number is a target not actual count"
})
print(f"4D. COMPLETED trials with ESTIMATED enrollment: {len(flagged)}")

4D. COMPLETED trials with ESTIMATED enrollment: 29


In [18]:
# VALIDATION SUMMARY
print("\n\nVALIDATION SUMMARY")
print("=" * 60)
for item in validation_log:
    print(f"\nCheck  : {item['check']}")
    print(f"Count  : {item['count']}")
    print(f"Action : {item['action']}")



VALIDATION SUMMARY

Check  : COMPLETED + 0 ACTUAL enrollment
Count  : 0
Action : Flag as suspicious — trial marked complete but no patients enrolled

Check  : primary_completion_date > completion_date
Count  : 1
Action : Flag as inconsistent — primary completion should never be after full completion

Check  : enrollment value present but type missing
Count  : 18
Action : Flag — cannot interpret enrollment number without knowing if actual or estimated

Check  : COMPLETED + ESTIMATED enrollment
Count  : 29
Action : Note — enrollment number is a target not actual count


In [19]:
# ─────────────────────────────────────────────────────────
# Add validation flags to dataframe
# ─────────────────────────────────────────────────────────

# Flag the one impossible date
df["flag_date_inconsistency"] = (
    (df["primary_completion_date"].notna()) &
    (df["completion_date"].notna()) &
    (df["primary_completion_date"] > df["completion_date"])
)

# Flag missing enrollment type
df["flag_missing_enrollment_type"] = (
    (df["enrollment_type"].isna()) &
    (df["enrollment"].notna())
)

# Flag completed with estimated enrollment
df["flag_estimated_enrollment_on_completed"] = (
    (df["recruitment_status"] == "COMPLETED") &
    (df["enrollment_type"] == "ESTIMATED")
)

print("Validation flags added.")
print(f"  flag_date_inconsistency                  : {df['flag_date_inconsistency'].sum()}")
print(f"  flag_missing_enrollment_type             : {df['flag_missing_enrollment_type'].sum()}")
print(f"  flag_estimated_enrollment_on_completed   : {df['flag_estimated_enrollment_on_completed'].sum()}")

Validation flags added.
  flag_date_inconsistency                  : 1
  flag_missing_enrollment_type             : 18
  flag_estimated_enrollment_on_completed   : 29


In [20]:
# ─────────────────────────────────────────────────────────
# SECTION 5 — CLEAN INTERVENTIONS
# ─────────────────────────────────────────────────────────

# Known non-drug intervention keywords
IMAGING = [
    "Computed Tomography", "Magnetic Resonance Imaging",
    "Positron Emission Tomography", "Bone Scan", "Echocardiography",
    "Contrast-Enhanced Ultrasound", "Single Photon Emission Computed Tomography",
    "Computed Tomography Colonography", "High-Resolution Microendoscopy",
    "Diagnostic Imaging", "Unenhanced ultrasound", "Fluorescence In Situ Hybridization",
    "Positron Emission Tomography and Computed Tomography Scan",
    "Positron Emission Tomography and Magnetic Resonance Imaging",
    "Near-Infrared Fluorescence Imaging", "Proflavine Hemisulfate",
    "18F-fluoroestradiol", "Gallium Ga-68 gozetotide",
    "Fludeoxyglucose F-18", "Gadopiclenol", "Gadoterate-meglumine",
    "Sulfur Hexafluoride Lipid Microspheres", "Indocyanine green"
]

SURGERY = [
    "Surgery", "Surgical Procedure", "Conventional Surgery",
    "Resection", "Hepatectomy", "Gastrectomy", "Esophagectomy",
    "Lobectomy", "Segmentectomy", "Cryosurgery", "Cryoablation",
    "Biopsy", "Bone Marrow Biopsy", "Bone Marrow Aspiration",
    "Transarterial Chemoembolization", "Cytoreductive Surgery",
    "Therapeutic Conventional Surgery", "Ablation Therapy",
    "Transurethral Resection of Bladder Tumor", "Laparoscopic Radical Prostatectomy with Robotics",
    "Wedge Excision", "Total Mesorectal Excision", "Peritonectomy",
    "Esophagogastrectomy", "Salvage Surgery", "Endoscopic Procedure",
    "Paracentesis", "Leukapheresis", "Lumbar Puncture",
    "Intrahepatic Infusion Procedure", "Intratumoral Route of Administration"
]

RADIATION = [
    "Radiation Therapy", "Stereotactic Body Radiation Therapy",
    "Intensity-Modulated Radiation Therapy", "Thoracic Radiation Therapy",
    "Whole-Brain Radiotherapy", "Brachytherapy", "Total-Body Irradiation",
    "External Beam Radiation Therapy", "Hypofractionated Radiation Therapy",
    "Low Dose Radiation Therapy", "Adjuvant Radiotherapy",
    "Cranial Irradiation", "Whole-Abdominal Irradiation",
    "3-Dimensional Conformal Radiation Therapy", "Accelerated Radiation Therapy",
    "Conventional Radiotherapy", "Normofractionated Radiotherapy",
    "Photon Beam Radiation Therapy", "Volume Modulated Arc Therapy",
    "Transarterial Radioembolization", "Total Marrow Irradiation",
    "High-Dose-Rate Vaginal Brachytherapy", "Chemoradiotherapy"
]

PLACEBO = [
    "Placebo Control", "Placebo", "Placebo Administration",
    "Vehicle Control", "Drug Vehicle"
]

DIAGNOSTIC = [
    "Laboratory Biomarker Analysis", "Biospecimen Collection",
    "Questionnaire Administration", "Gene Expression Analysis",
    "Flow Cytometry", "Next Generation Sequencing",
    "Immunohistochemistry Staining Method", "Bone Marrow Aspiration",
    "Polymerase Chain Reaction", "Cytogenetic Analysis",
    "Karyotyping", "Comparative Genomic Hybridization",
    "Genetic Variation Analysis", "Polymorphism Analysis",
    "TdT-Mediated dUTP Nick End Labeling Assay",
    "Reverse Transcriptase-Polymerase Chain Reaction",
    "Quality-of-Life Assessment", "Patient Observation",
    "Immunoenzyme Procedure"
]

TRANSPLANT = [
    "Hematopoietic Cell Transplantation",
    "Allogeneic Bone Marrow Transplantation",
    "Autologous Bone Marrow Transplantation",
    "Peripheral Blood Stem Cell Transplantation",
    "Umbilical Cord Blood Transplantation",
    "Allogeneic Hematopoietic Stem Cell Transplantation",
    "Autologous Hematopoietic Stem Cell Transplantation",
    "Allogeneic Stem Cell Transplantation",
    "In Vitro-Treated Peripheral Blood Stem Cell Transplantation",
    "Bone Marrow Transplantation", "Umbilical Cord Blood Stem Cell",
    "Autologous Islet Cell Transplantation",
    "Stem Cell Mobilization Therapy"
]

SUPPORTIVE = [
    "Resistance Training", "Counseling", "Best Practice",
    "Homeopathy Therapy", "Neoadjuvant Therapy",
    "Adjuvant Therapy", "Immunosuppressive Therapy",
    "Lymphodepletion Therapy", "Intrathecal Chemotherapy",
    "Total Parenteral Nutrition", "Blood Transfusion",
    "Bowel Preparation", "Physical prophylaxis",
    "Steroid Therapy", "Chemoradiotherapy"
]

print("Category dictionaries defined.")

Category dictionaries defined.


In [21]:
def categorise_intervention(drug_name):
    """
    Categorise a single intervention string into one of:
    imaging, surgery, radiation, placebo, diagnostic, 
    transplant, supportive, drug
    """
    name = str(drug_name).strip()
    
    if name in IMAGING:
        return "imaging"
    if name in SURGERY:
        return "surgery"
    if name in RADIATION:
        return "radiation"
    if name in PLACEBO:
        return "placebo"
    if name in DIAGNOSTIC:
        return "diagnostic"
    if name in TRANSPLANT:
        return "transplant"
    if name in SUPPORTIVE:
        return "supportive"
    return "drug"

# Apply to every intervention in every trial
# Result: a list of categories parallel to interventions_drugs
df["intervention_categories"] = df["interventions_drugs"].apply(
    lambda drugs: [categorise_intervention(d) for d in drugs]
)

# is_drug: True if at least one actual drug in the trial
df["has_drug"] = df["intervention_categories"].apply(
    lambda cats: any(c == "drug" for c in cats)
)

print("Intervention categories assigned.")
print("\nVerification — first 3 rows:")
for i in range(3):
    print(f"\n  Row {i}")
    print(f"  Drugs      : {df['interventions_drugs'].iloc[i]}")
    print(f"  Categories : {df['intervention_categories'].iloc[i]}")

Intervention categories assigned.

Verification — first 3 rows:

  Row 0
  Drugs      : ['Pembrolizumab', 'Lenvatinib']
  Categories : ['drug', 'drug']

  Row 1
  Drugs      : ['Tislelizumab']
  Categories : ['drug']

  Row 2
  Drugs      : ['Innocell']
  Categories : ['drug']


In [22]:
# How many interventions fall into each category across all trials?
from collections import Counter

all_categories = [
    cat 
    for cats in df["intervention_categories"] 
    for cat in cats
]

cat_counts = pd.Series(Counter(all_categories)).sort_values(ascending=False)

print("INTERVENTION CATEGORY DISTRIBUTION")
print("=" * 40)
print(cat_counts.to_string())

print(f"\nTrials with at least one drug : {df['has_drug'].sum()}")
print(f"Trials with NO drug           : {(~df['has_drug']).sum()}")

INTERVENTION CATEGORY DISTRIBUTION
drug          2328
radiation       87
diagnostic      72
imaging         62
surgery         62
placebo         55
transplant      32
supportive      18

Trials with at least one drug : 974
Trials with NO drug           : 26


In [23]:
# What are the 26 trials with no drug?
no_drug_trials = df[~df["has_drug"]]

print("TRIALS WITH NO DRUG INTERVENTION")
print("=" * 60)
print(f"Total: {len(no_drug_trials)}")
print()
print(no_drug_trials[["nct_id", "brief_title", 
                        "intervention_categories",
                        "recruitment_status",
                        "phase"]].to_string())

TRIALS WITH NO DRUG INTERVENTION
Total: 26

          nct_id                                                                                                                                           brief_title                                                  intervention_categories     recruitment_status          phase
20   NCT06417190                                                   Bladder Preservation for Patients with Muscle Invasive Bladder Cancer (MIBC) with Variant Histology                                                                       []             RECRUITING         PHASE2
45   NCT06261814                                                  Contrast Enhanced Ultrasound to Evaluate Response to Chemoembolization in Patients With Liver Tumors                                              [imaging, imaging, surgery]             RECRUITING         PHASE2
51   NCT06444412                             An Investigational Scan (Ga-68 PSMA-11 PET/CT) for Detection of Disease Recur

In [24]:
# ─────────────────────────────────────────────────────────
# SECTION 5B — DRUG NAME SYNONYM CLEANING
# ─────────────────────────────────────────────────────────

# Manual mapping for obvious synonyms and salt forms
DRUG_SYNONYMS = {
    # Typos
    "Cyclophosphamides"          : "Cyclophosphamide",
    
    # Salt forms → base drug
    "vincristine sulfate"        : "Vincristine",
    "doxorubicin hydrochloride"  : "Doxorubicin",
    "sunitinib malate"           : "Sunitinib",
    "imatinib mesylate"          : "Imatinib",
    "irinotecan hydrochloride"   : "Irinotecan",
    "fludarabine phosphate"      : "Fludarabine",
    "gemcitabine hydrochloride"  : "Gemcitabine",
    "pemetrexed disodium"        : "Pemetrexed",
    "bendamustine hydrochloride" : "Bendamustine",
    "topotecan hydrochloride"    : "Topotecan",
    "lapatinib ditosylate"       : "Lapatinib",
    "mitoxantrone hydrochloride" : "Mitoxantrone",
    "leucovorin calcium"         : "Folinic acid",
    "Levoleucovorin Calcium"     : "Folinic acid",
    "vinorelbine tartrate"       : "Vinorelbine",
    "vinblastine sulfate"        : "Vinblastine",
    "morphine sulfate"           : "Morphine",
    "osimertinib mesylate"       : "Osimertinib",
    "rivoceranib mesylate"       : "Rivoceranib",
    "catequentinib hydrochloride": "Catequentinib",
    "cabozantinib s-malate"      : "Cabozantinib",
    "pyrotinib dimaleate"        : "Pyrotinib",
    "sorafenib tosylate"         : "Sorafenib",
    "dovitinib lactate"          : "Dovitinib",
    "epirubicin hydrochloride"   : "Epirubicin",
    "daunorubicin hydrochloride" : "Daunorubicin",
    "bosutinib monohydrate"      : "Bosutinib",
    "enasidenib mesylate"        : "Enasidenib",
    "gilteritinib fumarate"      : "Gilteritinib",
    "eltrombopag olamine"        : "Eltrombopag",
    "glasdegib maleate"          : "Glasdegib",
    
    # Placebo variants
    "Placebo Control"            : "Placebo",
    "Placebo Administration"     : "Placebo",
    
    # Generic class names — flag but keep
    "PD-1 Inhibitor"             : "PD-1 Inhibitor (unspecified)",
    "PD-1/PD-L1 inhibitor"       : "PD-1/PD-L1 Inhibitor (unspecified)",
    "Chemotherapy"               : "Chemotherapy (unspecified)",
    "Immunotherapy"              : "Immunotherapy (unspecified)",
}

def clean_drug_name(drug_name):
    """Apply synonym mapping to a single drug name."""
    name = str(drug_name).strip()
    return DRUG_SYNONYMS.get(name, name)

# Apply to interventions_drugs
df["interventions_drugs_clean"] = df["interventions_drugs"].apply(
    lambda drugs: [clean_drug_name(d) for d in drugs]
)

print("Drug name synonyms cleaned.")
print("\nVerification — checking a few cleaned entries:")

# Show before/after for rows that had synonyms
changed = []
for idx, row in df.iterrows():
    original = row["interventions_drugs"]
    cleaned  = row["interventions_drugs_clean"]
    if original != cleaned:
        for o, c in zip(original, cleaned):
            if o != c:
                changed.append({"original": o, "cleaned": c})

changed_df = pd.DataFrame(changed).drop_duplicates()
print(changed_df.to_string(index=False))

Drug name synonyms cleaned.

Verification — checking a few cleaned entries:
                                                                original                                                                 cleaned
                                                         Placebo Control                                                                 Placebo
                                                            Chemotherapy                                              Chemotherapy (unspecified)
                                                           Immunotherapy                                             Immunotherapy (unspecified)
                                                    PD-1/PD-L1 inhibitor                                      PD-1/PD-L1 Inhibitor (unspecified)
                                                          PD-1 Inhibitor                                            PD-1 Inhibitor (unspecified)
                                                    os

In [25]:
# ─────────────────────────────────────────────────────────
# SECTION 6 — CLEAN INDICATIONS
# ─────────────────────────────────────────────────────────

# First — what does a typical indications list look like?
print("SAMPLE INDICATION LISTS")
print("=" * 60)
for i in [0, 1, 5, 10, 20, 50]:
    print(f"\nRow {i}: {df['indications'].iloc[i]}")

SAMPLE INDICATION LISTS

Row 0: ['Anorectal Cancer', 'Epithelial Neoplasm']

Row 1: ['Thoracic Neoplasm']

Row 5: ['Multiple Myeloma', 'Multiple Myeloma']

Row 10: ['Peritoneal Malignant Mesothelioma']

Row 20: ['Glioma', 'Muscle Invasive Bladder Cancer']

Row 50: ['Aplastic Anemia', 'Transfusion-dependent Anemia']


In [26]:
# ─────────────────────────────────────────────────────────
# SECTION 6A — VAGUE TERMS TO SKIP
# ─────────────────────────────────────────────────────────

VAGUE_TERMS = {
    "Solid Tumors",
    "Epithelial Neoplasm",
    "Digestive System Neoplasm",
    "Hematology",
    "Neoplasm",
    "Cancer",
    "Malignant Neoplasm",
    "Advanced Cancer",
    "Metastatic Cancer",
    "Unresectable Solid Tumor",
    "Locally Advanced Solid Tumor",
    "Carcinoma",
    "Tumor",
    "Malignancy"
}

print(f"Vague terms defined: {len(VAGUE_TERMS)}")

Vague terms defined: 14


In [27]:
# ─────────────────────────────────────────────────────────
# SECTION 6B — BROAD CATEGORY MAPPING
# ─────────────────────────────────────────────────────────

BROAD_CATEGORY_MAP = {
    # Lung
    "Non-Small Cell Lung Cancer"                    : "Lung Cancer",
    "Small Cell Lung Cancer"                        : "Lung Cancer",
    "Thoracic Neoplasm"                             : "Lung Cancer",
    "Lung Cancer"                                   : "Lung Cancer",
    "Lung Non-Squamous Non-Small Cell Carcinoma"    : "Lung Cancer",
    "Lung Squamous Cell Carcinoma"                  : "Lung Cancer",

    # Breast
    "Breast Cancer"                                 : "Breast Cancer",
    "Breast Neoplasm"                               : "Breast Cancer",
    "Triple-Negative Breast Cancer"                 : "Breast Cancer",
    "HER2-Negative Breast Cancer"                   : "Breast Cancer",
    "Hormone Receptor-Positive Breast Cancer"       : "Breast Cancer",
    "Inflammatory Breast Cancer"                    : "Breast Cancer",

    # Blood — separated properly as you pointed out
    "Multiple Myeloma"                              : "Multiple Myeloma",
    "Plasma Cell Leukemia"                          : "Multiple Myeloma",
    "Acute Myeloid Leukemia"                        : "Leukemia",
    "Acute Lymphoblastic Leukemia"                  : "Leukemia",
    "Leukemia"                                      : "Leukemia",
    "Chronic Lymphocytic Leukemia"                  : "Leukemia",
    "T-cell acute lymphoblastic leukemia"           : "Leukemia",
    "Diffuse Large B-Cell Lymphoma"                 : "Lymphoma",
    "Non-Hodgkin Lymphoma"                          : "Lymphoma",
    "Follicular Lymphoma"                           : "Lymphoma",
    "Lymphoma"                                      : "Lymphoma",
    "Hematology"                                    : "Blood Cancer (unspecified)",
    "Myelodysplastic Syndrome"                      : "Myelodysplastic Syndrome",
    "Primary Myelofibrosis"                         : "Myelofibrosis",

    # Colorectal
    "Colorectal Cancer"                             : "Colorectal Cancer",
    "Colon Carcinoma"                               : "Colorectal Cancer",
    "Rectal Carcinoma"                              : "Colorectal Cancer",
    "Colorectal Carcinoma"                          : "Colorectal Cancer",

    # Pancreatic
    "Pancreatic Cancer"                             : "Pancreatic Cancer",
    "Pancreatic Ductal Adenocarcinoma"              : "Pancreatic Cancer",

    # Prostate
    "Prostate Cancer"                               : "Prostate Cancer",
    "Castration-Resistant Prostate Carcinoma"       : "Prostate Cancer",

    # Head & Neck
    "Head and Neck Squamous Cell Carcinoma"         : "Head & Neck Cancer",
    "Head and Neck Neoplasm"                        : "Head & Neck Cancer",
    "Nasopharyngeal Cancer"                         : "Head & Neck Cancer",
    "Oral Cancer"                                   : "Head & Neck Cancer",

    # Liver
    "Hepatocellular Carcinoma"                      : "Liver Cancer",
    "Intrahepatic Cholangiocarcinoma"               : "Liver Cancer",
    "Liver Cancer"                                  : "Liver Cancer",
    "Bile Duct Cancer"                              : "Liver Cancer",

    # Gynecologic
    "Ovarian Cancer"                                : "Gynecologic Cancer",
    "Cervical Cancer"                               : "Gynecologic Cancer",
    "Endometrial Cancer"                            : "Gynecologic Cancer",
    "Vulvar Carcinoma"                              : "Gynecologic Cancer",
    "Fallopian Tube Cancer"                         : "Gynecologic Cancer",
    "Primary Peritoneal Cancer"                     : "Gynecologic Cancer",

    # Gastric
    "Gastric Cancer"                                : "Gastric Cancer",
    "Stomach Neoplasm"                              : "Gastric Cancer",
    "Gastric Carcinoma"                             : "Gastric Cancer",

    # Bladder
    "Bladder Cancer"                                : "Bladder Cancer",
    "Muscle Invasive Bladder Cancer"                : "Bladder Cancer",

    # Skin
    "Melanoma"                                      : "Skin Cancer",
    "Skin Melanoma"                                 : "Skin Cancer",
    "Skin Squamous Cell Carcinoma"                  : "Skin Cancer",

    # Brain
    "Glioblastoma"                                  : "Brain Cancer",
    "Glioma"                                        : "Brain Cancer",
    "Central Nervous System Neoplasm"               : "Brain Cancer",

    # Sarcoma
    "Soft Tissue Sarcoma"                           : "Sarcoma",
    "Osteosarcoma"                                  : "Sarcoma",

    # Esophageal
    "Esophageal Cancer"                             : "Esophageal Cancer",

    # Non-cancer
    "Sickle Cell Disease"                           : "Non-Cancer",
    "Aplastic Anemia"                               : "Non-Cancer",
    "Inflammatory Bowel Disease"                    : "Non-Cancer",
    "Iron-Deficiency Anemia"                        : "Non-Cancer",
    "Warm Autoimmune Hemolytic Anemia"              : "Non-Cancer",
    "Chronic Graft Versus Host Disease"             : "Non-Cancer",
    "Graft Versus Host Disease"                     : "Non-Cancer",
    "Anemia"                                        : "Non-Cancer",
}

print(f"Broad category map defined: {len(BROAD_CATEGORY_MAP)} entries")

Broad category map defined: 70 entries


In [28]:
# ─────────────────────────────────────────────────────────
# SECTION 6C — APPLY TO DATASET
# ─────────────────────────────────────────────────────────

def get_primary_indication(indication_list):
    """
    Pick the most specific meaningful indication.
    Skip vague umbrella terms.
    Fall back to first item if nothing specific found.
    """
    if not indication_list:
        return "Unknown"
    
    # Remove duplicates while preserving order
    seen = set()
    unique = []
    for ind in indication_list:
        if ind not in seen:
            seen.add(ind)
            unique.append(ind)
    
    # Try to find first non-vague term
    for ind in unique:
        if ind not in VAGUE_TERMS:
            return ind
    
    # If all are vague, return first
    return unique[0]

def get_broad_category(primary_indication):
    """
    Map primary indication to broad category.
    Use fuzzy matching as fallback for unmapped values.
    """
    # Direct map first
    if primary_indication in BROAD_CATEGORY_MAP:
        return BROAD_CATEGORY_MAP[primary_indication]
    
    # Fuzzy match against known keys
    match, score, _ = process.extractOne(
        primary_indication,
        BROAD_CATEGORY_MAP.keys(),
        scorer=fuzz.token_sort_ratio
    )
    
    # Only accept high confidence matches
    if score >= 80:
        return BROAD_CATEGORY_MAP[match]
    
    return "Other Cancer"

# Apply both functions
df["primary_indication"] = df["indications"].apply(get_primary_indication)
df["broad_category"]     = df["primary_indication"].apply(get_broad_category)

print("PRIMARY INDICATION & BROAD CATEGORY ASSIGNED")
print("=" * 60)
print(f"\nUnique primary indications : {df['primary_indication'].nunique()}")
print(f"Unique broad categories    : {df['broad_category'].nunique()}")

print("\n\nBROAD CATEGORY DISTRIBUTION")
print("=" * 40)
print(df["broad_category"].value_counts().to_string())

PRIMARY INDICATION & BROAD CATEGORY ASSIGNED

Unique primary indications : 217
Unique broad categories    : 22


BROAD CATEGORY DISTRIBUTION
broad_category
Other Cancer                  345
Breast Cancer                  84
Lung Cancer                    78
Leukemia                       72
Multiple Myeloma               40
Head & Neck Cancer             40
Non-Cancer                     33
Prostate Cancer                33
Lymphoma                       33
Colorectal Cancer              33
Gynecologic Cancer             32
Pancreatic Cancer              31
Liver Cancer                   28
Brain Cancer                   25
Gastric Cancer                 22
Blood Cancer (unspecified)     15
Esophageal Cancer              14
Myelodysplastic Syndrome       13
Bladder Cancer                  9
Sarcoma                         8
Skin Cancer                     8
Myelofibrosis                   4


In [29]:
# What indications are falling into Other Cancer?
other = df[df["broad_category"] == "Other Cancer"]["primary_indication"].value_counts()
print(f"Indications mapped to Other Cancer: {other.nunique()}")
print()
print(other.head(30).to_string())

Indications mapped to Other Cancer: 12

primary_indication
Solid Tumors                                 52
Epithelial Neoplasm                          17
Digestive System Neoplasm                    15
Neoplasms                                    11
Male Reproductive System Neoplasm            11
Cancer                                       11
Brain Metastasis                              9
Uveal Melanoma                                7
Female Reproductive System Neoplasm           6
Renal Cell Carcinoma                          5
Neuroendocrine Tumors                         5
Extensive Stage Lung Small Cell Carcinoma     4
Tumors                                        4
Non-Clear Cell Renal Cell Carcinoma           4
Urothelial Carcinoma                          4
Biliary Tract Cancer                          4
Thyroid Cancer                                4
Hodgkin Lymphoma                              4
Actinic Keratosis                             4
Sarcoma                      

In [30]:
# Fix 1 — Add missing cancers to map
BROAD_CATEGORY_MAP.update({
    "Renal Cell Carcinoma"                      : "Kidney Cancer",
    "Clear Cell Renal Cell Carcinoma"           : "Kidney Cancer",
    "Non-Clear Cell Renal Cell Carcinoma"       : "Kidney Cancer",
    "Kidney Cancer"                             : "Kidney Cancer",
    "Kidney Wilms Tumor"                        : "Kidney Cancer",
    "Neuroendocrine Tumors"                     : "Neuroendocrine Cancer",
    "Urothelial Carcinoma"                      : "Bladder Cancer",
    "Thyroid Cancer"                            : "Thyroid Cancer",
    "Hodgkin Lymphoma"                          : "Lymphoma",
    "Chronic Myeloid Leukemia"                  : "Leukemia",
    "Biliary Tract Cancer"                      : "Liver Cancer",
    "Uveal Melanoma"                            : "Skin Cancer",
    "Metastatic Melanoma"                       : "Skin Cancer",
    "Malignant Melanoma"                        : "Skin Cancer",
    "High-Grade Glioma"                         : "Brain Cancer",
    "Brain Metastasis"                          : "Brain Cancer",
    "Plasma Cell Neoplasm"                      : "Multiple Myeloma",
    "Connective and Soft Tissue Neoplasm"       : "Sarcoma",
    "Intraepithelial Neoplasia"                 : "Other Cancer",
    "Actinic Keratosis"                         : "Skin Cancer",
    "Extensive Stage Lung Small Cell Carcinoma" : "Lung Cancer",

    # Fix 2 — Vague terms that become primary when nothing better exists
    # These should be labelled Unspecified
    "Solid Tumors"                              : "Unspecified",
    "Epithelial Neoplasm"                       : "Unspecified",
    "Digestive System Neoplasm"                 : "Unspecified",
    "Neoplasms"                                 : "Unspecified",
    "Cancer"                                    : "Unspecified",
    "Tumors"                                    : "Unspecified",
    "Male Reproductive System Neoplasm"         : "Unspecified",
    "Female Reproductive System Neoplasm"       : "Unspecified",
})

# Also add Sarcoma to map (it was showing as primary but mapping to Other)
BROAD_CATEGORY_MAP.update({
    "Sarcoma" : "Sarcoma",
})

print(f"Updated map: {len(BROAD_CATEGORY_MAP)} entries")

Updated map: 100 entries


In [31]:
# Reapply both functions with updated map
df["primary_indication"] = df["indications"].apply(get_primary_indication)
df["broad_category"]     = df["primary_indication"].apply(get_broad_category)

print("UPDATED BROAD CATEGORY DISTRIBUTION")
print("=" * 40)
print(df["broad_category"].value_counts().to_string())

# Check what's still in Other Cancer
other = df[df["broad_category"] == "Other Cancer"]["primary_indication"].value_counts()
print(f"\nRemaining in Other Cancer: {other.sum()} trials, {other.nunique()} unique indications")
if len(other) > 0:
    print(other.to_string())

UPDATED BROAD CATEGORY DISTRIBUTION
broad_category
Other Cancer                  128
Unspecified                   127
Breast Cancer                  84
Lung Cancer                    82
Leukemia                       78
Multiple Myeloma               43
Head & Neck Cancer             40
Lymphoma                       38
Brain Cancer                   37
Colorectal Cancer              33
Prostate Cancer                33
Non-Cancer                     33
Gynecologic Cancer             32
Liver Cancer                   32
Pancreatic Cancer              31
Skin Cancer                    25
Gastric Cancer                 22
Kidney Cancer                  18
Sarcoma                        15
Blood Cancer (unspecified)     15
Esophageal Cancer              14
Myelodysplastic Syndrome       13
Bladder Cancer                 13
Thyroid Cancer                  5
Neuroendocrine Cancer           5
Myelofibrosis                   4

Remaining in Other Cancer: 128 trials, 3 unique indications
prim

In [32]:
import requests
import json

url = "https://oncotree.mskcc.org/api/tumorTypes?format=csv"
response = requests.get(url)
data = response.json()

# Convert to dataframe
oncotree_df = pd.DataFrame(data)
print(oncotree_df.shape)
print(oncotree_df.columns.tolist())
print(oncotree_df.head())

(897, 12)
['code', 'color', 'name', 'mainType', 'externalReferences', 'tissue', 'children', 'parent', 'history', 'level', 'revocations', 'precursors']
            code           color           name              mainType  \
0          LIVER  MediumSeaGreen          Liver          Liver Cancer   
1       PANCREAS          Purple       Pancreas     Pancreatic Cancer   
2           BONE           White           Bone           Bone Cancer   
3  ADRENAL_GLAND          Purple  Adrenal Gland  Adrenal Gland Cancer   
4          PENIS            Blue          Penis         Penile Cancer   

                          externalReferences         tissue children  parent  \
0  {'UMLS': ['C0023884'], 'NCI': ['C12392']}          Liver       {}  TISSUE   
1  {'UMLS': ['C0030274'], 'NCI': ['C12393']}       Pancreas       {}  TISSUE   
2  {'UMLS': ['C0262950'], 'NCI': ['C12366']}           Bone       {}  TISSUE   
3  {'UMLS': ['C0001625'], 'NCI': ['C12666']}  Adrenal Gland       {}  TISSUE   
4  {'UMLS'

In [33]:
# ─────────────────────────────────────────────────────────
# SECTION 6D — ONCOTREE INTERSECTION
# ─────────────────────────────────────────────────────────

# Keep only relevant columns
oncotree_clean = oncotree_df[["name", "mainType", "tissue"]].copy()
oncotree_clean.columns = ["oncotree_name", "broad_category_oncotree", "tissue"]

print(f"OncoTree entries: {len(oncotree_clean)}")
print(f"\nSample mainTypes:")
print(oncotree_clean["broad_category_oncotree"].value_counts().head(20).to_string())

OncoTree entries: 897

Sample mainTypes:
broad_category_oncotree
Mature B-Cell Neoplasms       56
Soft Tissue Sarcoma           49
Leukemia                      43
Germ Cell Tumor               34
Mature T and NK Neoplasms     31
Breast Cancer                 28
Non-Small Cell Lung Cancer    27
Ovarian Cancer                26
Cervical Cancer               24
Bone Cancer                   24
Skin Cancer, Non-Melanoma     20
Glioma                        20
Esophagogastric Cancer        19
Head and Neck Cancer          18
Renal Cell Carcinoma          18
Melanoma                      17
Hepatobiliary Cancer          17
Bladder Cancer                16
Pancreatic Cancer             16
CNS Cancer                    15


In [34]:
# Create a lookup dictionary from OncoTree
# name → broad_category
oncotree_lookup = dict(zip(
    oncotree_clean["oncotree_name"].str.lower(),
    oncotree_clean["broad_category_oncotree"]
))

print(f"OncoTree lookup entries: {len(oncotree_lookup)}")

# Try exact match first, then fuzzy match
def get_broad_category_v2(primary_indication):
    if not primary_indication or primary_indication == "Unknown":
        return "Unknown"
    
    ind_lower = primary_indication.lower()
    
    # 1. Check our manual map first (highest priority)
    if primary_indication in BROAD_CATEGORY_MAP:
        return BROAD_CATEGORY_MAP[primary_indication]
    
    # 2. Exact match against OncoTree
    if ind_lower in oncotree_lookup:
        return oncotree_lookup[ind_lower]
    
    # 3. Fuzzy match against OncoTree names
    match, score, _ = process.extractOne(
        ind_lower,
        oncotree_lookup.keys(),
        scorer=fuzz.token_sort_ratio
    )
    if score >= 85:
        return oncotree_lookup[match]
    
    # 4. Fall back to Other Cancer
    return "Other Cancer"

# Reapply with OncoTree
df["broad_category"] = df["primary_indication"].apply(get_broad_category_v2)

print("\nUPDATED BROAD CATEGORY DISTRIBUTION (with OncoTree)")
print("=" * 50)
print(df["broad_category"].value_counts().to_string())

# Check remaining Other Cancer
other = df[df["broad_category"] == "Other Cancer"]["primary_indication"].value_counts()
print(f"\nRemaining in Other Cancer: {other.sum()} trials")
print(other.head(20).to_string())

OncoTree lookup entries: 879

UPDATED BROAD CATEGORY DISTRIBUTION (with OncoTree)
broad_category
Unspecified                                     127
Other Cancer                                    118
Lung Cancer                                      82
Breast Cancer                                    79
Leukemia                                         76
Multiple Myeloma                                 43
Head & Neck Cancer                               39
Brain Cancer                                     36
Colorectal Cancer                                33
Prostate Cancer                                  33
Gynecologic Cancer                               32
Pancreatic Cancer                                31
Liver Cancer                                     31
Non-Cancer                                       30
Lymphoma                                         29
Skin Cancer                                      25
Gastric Cancer                                   22
Kidney Cancer      

In [35]:
# Consolidate OncoTree categories into our standard buckets
CONSOLIDATE_MAP = {
    "Non-Hodgkin Lymphoma"                    : "Lymphoma",
    "Hodgkin Lymphoma"                        : "Lymphoma",
    "Mature B-Cell Neoplasms"                 : "Lymphoma",
    "Mature T and NK Neoplasms"               : "Blood Cancer (unspecified)",
    "Soft Tissue Sarcoma"                     : "Sarcoma",
    "Melanoma"                                : "Skin Cancer",
    "Skin Cancer, Non-Melanoma"               : "Skin Cancer",
    "Myeloproliferative Neoplasms"            : "Myelofibrosis",
    "Hepatobiliary Cancer"                    : "Liver Cancer",
    "Posttransplant Lymphoproliferative Disorders" : "Lymphoma",
    "Embryonal Tumor"                         : "Brain Cancer",
    "Miscellaneous Brain Tumor"               : "Brain Cancer",
    "Small Bowel Cancer"                      : "Other Cancer",
    "Gastrointestinal Stromal Tumor"          : "Sarcoma",
    "Cancer of Unknown Primary"               : "Other Cancer",
    "Germ Cell Tumor"                         : "Other Cancer",
    "Parathyroid Cancer"                      : "Other Cancer",
}

df["broad_category"] = df["broad_category"].replace(CONSOLIDATE_MAP)

In [36]:
BROAD_CATEGORY_MAP.update({
    "HER2-Positive Breast Cancer"                       : "Breast Cancer",
    "Metastatic NSCLC"                                  : "Lung Cancer",
    "Multiple Primary Lung Cancers"                     : "Lung Cancer",
    "Anorectal Cancer"                                  : "Colorectal Cancer",
    "Anal Cancer"                                       : "Colorectal Cancer",
    "WHO Grade 3 Glioma"                                : "Brain Cancer",
    "WHO Grade 4 Glioma"                                : "Brain Cancer",
    "Anaplastic Astrocytoma"                            : "Brain Cancer",
    "Chronic Leukemia"                                  : "Leukemia",
    "T Lymphoblastic Lymphoma"                          : "Lymphoma",
    "Peripheral T-Cell Lymphoma, Not Otherwise Specified" : "Lymphoma",
    "Urinary System Neoplasm"                           : "Bladder Cancer",
    "Neurofibromatosis"                                 : "Other Cancer",
    "Acute Graft Versus Host Disease"                   : "Non-Cancer",
    "Paroxysmal Nocturnal Hemoglobinuria"               : "Non-Cancer",
    "Malignant Ascites"                                 : "Other Cancer",
    
    # Data entry errors — drug and procedure names in indication field
    "Furmonertinib"                                     : "Unknown — Data Entry Error",
    "Hepatectomy"                                       : "Unknown — Data Entry Error",
})

# Reapply
df["broad_category"] = df["primary_indication"].apply(get_broad_category_v2)
df["broad_category"] = df["broad_category"].replace(CONSOLIDATE_MAP)

print("FINAL BROAD CATEGORY DISTRIBUTION")
print("=" * 50)
print(df["broad_category"].value_counts().to_string())

remaining = df[df["broad_category"] == "Other Cancer"]["primary_indication"].value_counts()
print(f"\nRemaining in Other Cancer: {remaining.sum()} trials")

FINAL BROAD CATEGORY DISTRIBUTION
broad_category
Unspecified                   127
Other Cancer                   96
Breast Cancer                  84
Lung Cancer                    84
Leukemia                       78
Multiple Myeloma               43
Lymphoma                       43
Brain Cancer                   42
Head & Neck Cancer             39
Colorectal Cancer              36
Non-Cancer                     33
Prostate Cancer                33
Gynecologic Cancer             32
Liver Cancer                   32
Pancreatic Cancer              31
Skin Cancer                    28
Gastric Cancer                 22
Blood Cancer (unspecified)     19
Sarcoma                        18
Kidney Cancer                  18
Bladder Cancer                 15
Esophageal Cancer              14
Myelodysplastic Syndrome       13
Myelofibrosis                   8
Neuroendocrine Cancer           5
Thyroid Cancer                  4
Unknown — Data Entry Error      3

Remaining in Other Cancer: 96 tr

In [37]:
BROAD_CATEGORY_MAP.update({
    "HER2-Positive Breast Cancer"                       : "Breast Cancer",
    "Metastatic NSCLC"                                  : "Lung Cancer",
    "Multiple Primary Lung Cancers"                     : "Lung Cancer",
    "Anorectal Cancer"                                  : "Colorectal Cancer",
    "Anal Cancer"                                       : "Colorectal Cancer",
    "WHO Grade 3 Glioma"                                : "Brain Cancer",
    "WHO Grade 4 Glioma"                                : "Brain Cancer",
    "Anaplastic Astrocytoma"                            : "Brain Cancer",
    "Chronic Leukemia"                                  : "Leukemia",
    "T Lymphoblastic Lymphoma"                          : "Lymphoma",
    "Peripheral T-Cell Lymphoma, Not Otherwise Specified" : "Lymphoma",
    "Urinary System Neoplasm"                           : "Bladder Cancer",
    "Neurofibromatosis"                                 : "Other Cancer",
    "Acute Graft Versus Host Disease"                   : "Non-Cancer",
    "Paroxysmal Nocturnal Hemoglobinuria"               : "Non-Cancer",
    "Malignant Ascites"                                 : "Other Cancer",
    
    # Data entry errors — drug and procedure names in indication field
    "Furmonertinib"                                     : "Unknown — Data Entry Error",
    "Hepatectomy"                                       : "Unknown — Data Entry Error",
})

# Reapply
df["broad_category"] = df["primary_indication"].apply(get_broad_category_v2)
df["broad_category"] = df["broad_category"].replace(CONSOLIDATE_MAP)

print("FINAL BROAD CATEGORY DISTRIBUTION")
print("=" * 50)
print(df["broad_category"].value_counts().to_string())

remaining = df[df["broad_category"] == "Other Cancer"]["primary_indication"].value_counts()
print(f"\nRemaining in Other Cancer: {remaining.sum()} trials")

FINAL BROAD CATEGORY DISTRIBUTION
broad_category
Unspecified                   127
Other Cancer                   96
Breast Cancer                  84
Lung Cancer                    84
Leukemia                       78
Multiple Myeloma               43
Lymphoma                       43
Brain Cancer                   42
Head & Neck Cancer             39
Colorectal Cancer              36
Non-Cancer                     33
Prostate Cancer                33
Gynecologic Cancer             32
Liver Cancer                   32
Pancreatic Cancer              31
Skin Cancer                    28
Gastric Cancer                 22
Blood Cancer (unspecified)     19
Sarcoma                        18
Kidney Cancer                  18
Bladder Cancer                 15
Esophageal Cancer              14
Myelodysplastic Syndrome       13
Myelofibrosis                   8
Neuroendocrine Cancer           5
Thyroid Cancer                  4
Unknown — Data Entry Error      3

Remaining in Other Cancer: 96 tr

In [38]:
total = len(df)
properly_mapped = df[~df["broad_category"].isin([
    "Other Cancer", "Unspecified", "Unknown — Data Entry Error", "Unknown"
])].shape[0]

print(f"Total trials          : {total}")
print(f"Properly categorised  : {properly_mapped} ({properly_mapped/total*100:.1f}%)")
print(f"Other/Unspecified     : {total - properly_mapped} ({(total-properly_mapped)/total*100:.1f}%)")

Total trials          : 1000
Properly categorised  : 774 (77.4%)
Other/Unspecified     : 226 (22.6%)


In [39]:
# ─────────────────────────────────────────────────────────
# SECTION 7 — PRIMARY TECHNOLOGY & PRIMARY TARGET
# ─────────────────────────────────────────────────────────

def get_primary_technology(tech_list):
    """
    Extract first valid technology from nested list.
    main_technologies is a list of lists — one inner list per drug.
    """
    if not tech_list:
        return "Unknown"
    for inner in tech_list:
        if isinstance(inner, list) and len(inner) > 0:
            return inner[0]
    return "Unknown"

def get_primary_target(target_list):
    """
    Extract first valid target abbreviation from nested list.
    """
    if not target_list:
        return "Unknown"
    for inner in target_list:
        if isinstance(inner, list) and len(inner) > 0:
            return inner[0]
    return "Unknown"

df["primary_technology"] = df["main_technologies"].apply(get_primary_technology)
df["primary_target"]     = df["target_abbreviations"].apply(get_primary_target)

print("PRIMARY TECHNOLOGY DISTRIBUTION")
print("=" * 40)
print(df["primary_technology"].value_counts().to_string())

print("\n\nPRIMARY TARGET DISTRIBUTION (top 20)")
print("=" * 40)
print(df["primary_target"].value_counts().head(20).to_string())

PRIMARY TECHNOLOGY DISTRIBUTION
primary_technology
Small Molecule                                                             566
Antibody                                                                   168
Unknown                                                                    124
Other Protein Therapy                                                       34
Engineered Protein Therapy                                                  17
Antibody Drug Conjugate (ADC)                                               16
Chimeric Antigen Receptor T-Cell Therapy (CAR-T)                            13
Radiopharmaceutical Imaging                                                 12
Imaging                                                                      6
Cancer Vaccine                                                               5
Subunit Vaccine                                                              4
Cell Therapy                                                                 4
L

In [40]:
# ─────────────────────────────────────────────────────────
# SECTION 8 — FIX ENCODING IN TARGET ABBREVIATIONS
# ─────────────────────────────────────────────────────────

# Fix corrupted Greek letters in primary_target
ENCODING_FIX = {
    "Î±" : "α",
    "Î²" : "β",
    "Î³" : "γ",
    "Î´" : "δ",
    "Â²" : "²",
    "Â¹" : "¹",
    "â…¡" : "Ⅱ",
    "â…¢" : "Ⅲ",
}

def fix_encoding(text):
    if not isinstance(text, str):
        return text
    for bad, good in ENCODING_FIX.items():
        text = text.replace(bad, good)
    return text

df["primary_target"] = df["primary_target"].apply(fix_encoding)

print("ENCODING FIXED — PRIMARY TARGET (top 20)")
print("=" * 40)
print(df["primary_target"].value_counts().head(20).to_string())

ENCODING FIXED — PRIMARY TARGET (top 20)
primary_target
Unknown          160
DNA               97
PD-1              53
TUBB1             45
EGFR              26
TYMS              24
PDL1              22
GR                18
CD20              14
VEGF-A            11
FGFR1             11
BCR-ABL1          11
Proteasome        11
Beta-tubulin       9
CRAF               9
TOP1               8
ER                 8
MEK1               8
CDK4               8
PI3Kα              8


In [41]:
# ─────────────────────────────────────────────────────────
# SECTION 9 — OUTPUT CLEAN DATASET
# ─────────────────────────────────────────────────────────

# Define final columns to keep
FINAL_COLS = [
    # Identifiers
    "ID-datalake", "nct_id", "brief_title",
    
    # Phase
    "phase", "phase_int", "phase_numeric", "trial_type",
    
    # Status
    "recruitment_status",
    
    # Dates
    "start_date", "completion_date", "primary_completion_date",
    "start_year", "trial_duration_days",
    
    # Enrollment
    "enrollment", "enrollment_type",
    
    # Validation flags
    "flag_date_inconsistency",
    "flag_missing_enrollment_type",
    "flag_estimated_enrollment_on_completed",
    
    # Indications
    "indications", "primary_indication", "broad_category",
    
    # Interventions
    "interventions_drugs_clean", "intervention_categories",
    "has_drug",
    
    # Technologies
    "main_technologies", "primary_technology",
    
    # Targets
    "target_abbreviations", "primary_target",
    
    # Drugs datalake
    "drugs_datalake"
]

df_clean = df[FINAL_COLS].copy()

print(f"Clean dataset shape: {df_clean.shape}")
print(f"\nColumns: {df_clean.columns.tolist()}")

Clean dataset shape: (1000, 29)

Columns: ['ID-datalake', 'nct_id', 'brief_title', 'phase', 'phase_int', 'phase_numeric', 'trial_type', 'recruitment_status', 'start_date', 'completion_date', 'primary_completion_date', 'start_year', 'trial_duration_days', 'enrollment', 'enrollment_type', 'flag_date_inconsistency', 'flag_missing_enrollment_type', 'flag_estimated_enrollment_on_completed', 'indications', 'primary_indication', 'broad_category', 'interventions_drugs_clean', 'intervention_categories', 'has_drug', 'main_technologies', 'primary_technology', 'target_abbreviations', 'primary_target', 'drugs_datalake']


In [42]:
# Save to CSV
df_clean.to_csv("data/df_clean.csv", index=False)

print("df_clean.csv saved successfully.")
print(f"\nFinal shape : {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")
print(f"\nColumn summary:")
for col in df_clean.columns:
    null_count = df_clean[col].isna().sum()
    print(f"  {col:<45} {null_count} nulls")

df_clean.csv saved successfully.

Final shape : 1000 rows x 29 columns

Column summary:
  ID-datalake                                   0 nulls
  nct_id                                        0 nulls
  brief_title                                   0 nulls
  phase                                         40 nulls
  phase_int                                     40 nulls
  phase_numeric                                 40 nulls
  trial_type                                    0 nulls
  recruitment_status                            0 nulls
  start_date                                    5 nulls
  completion_date                               52 nulls
  primary_completion_date                       51 nulls
  start_year                                    5 nulls
  trial_duration_days                           54 nulls
  enrollment                                    26 nulls
  enrollment_type                               44 nulls
  flag_date_inconsistency                       0 nulls
  flag_m

In [43]:
# Check all rows with [[]] in target_names
# What do their interventions look like?

mask = df_raw["target_names"] == "[[]]"
subset = df[mask][["nct_id", "interventions_drugs_clean", 
                    "intervention_categories", "has_drug"]].copy()

print(f"Total rows with [[]] in target_names: {len(subset)}")
print()
print("HAS_DRUG distribution:")
print(subset["has_drug"].value_counts().to_string())
print()
print("Sample rows WITH drug but no target:")
has_drug = subset[subset["has_drug"] == True]
print(f"Count: {len(has_drug)}")
print(has_drug[["nct_id", "interventions_drugs_clean"]].head(10).to_string())
print()
print("Sample rows with NO drug:")
no_drug = subset[subset["has_drug"] == False]
print(f"Count: {len(no_drug)}")
print(no_drug[["nct_id", "interventions_drugs_clean"]].head(10).to_string())

Total rows with [[]] in target_names: 86

HAS_DRUG distribution:
has_drug
True     80
False     6

Sample rows WITH drug but no target:
Count: 80
          nct_id                                                                         interventions_drugs_clean
2    NCT06366490                                                                                        [Innocell]
28   NCT06772428                                [Megestrol acetate, Radiation Therapy, Chemotherapy (unspecified)]
40   NCT06613152                                                                                        [HS-IT201]
50   NCT06525948                                                   [Recombinant Human Thrombopoietin, Cyclosporin]
59   NCT06402383                                                                    [Human papillomavirus vaccine]
79   NCT05995041  [CLL-1/CD33/CD38/CD123-specific universal CAR- T cells (Shenzhen Geno-Immune Medical Institute)]
85   NCT06321887                                 